In [1]:
import io
import re
import requests
import openpyxl
from bs4 import BeautifulSoup
from datetime import date
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept-Language': 'pl-PL,pl;q=0.9'
}
DATA_POBRANIA = date.today().strftime('%Y-%m-%d')


# --- Funkcje pomocnicze ---

def get_soup(url):
    odpowiedz = requests.get(url, headers=HEADERS)
    return BeautifulSoup(odpowiedz.text, 'html.parser')

def na_float(tekst):
    """Zamienia tekst z cena (np. '1,95 PLN/kWh') na liczbe."""
    tylko_liczba = re.sub(r'[^\d,\.]', '', str(tekst)).replace(',', '.')
    try:
        return float(tylko_liczba)
    except ValueError:
        return 0.0

def format_widel(lista_cen):
    """Zwraca '1,96 PLN/kWh' dla jednej ceny albo '2,35-2,66 PLN/kWh' dla widełek."""
    unikalne = sorted(set(c for c in lista_cen if c > 0))
    if not unikalne:
        return 0.0
    if len(unikalne) == 1:
        return unikalne[0]
    return f'{unikalne[0]:.2f}-{unikalne[-1]:.2f} PLN/kWh'.replace('.', ',')

def darmowy_czas(tekst_postoju):
    """Wyciaga liczbe minut z tekstu postoju, np. '0,40 PLN/min po 180 min' -> '180 min'."""
    wynik = re.search(r'po\s*(\d+)\s*min', str(tekst_postoju), re.I)
    if wynik:
        return wynik.group(1) + ' min'
    return 'brak danych'


# ==========================================
# 1. ORLEN CHARGE
# Zrodlo: orlencharge.pl/cennik/ (tabele HTML)
# ==========================================
def scrape_orlen():
    url = 'https://orlencharge.pl/cennik/'
    soup = get_soup(url)
    tabele = soup.find_all('table')
    if len(tabele) < 2:
        raise ValueError('Nie znaleziono tabel na stronie ORLEN')

    # Tabela 1: ceny AC/DC
    ceny = {}
    for wiersz in tabele[0].find_all('tr')[1:]:
        komorki = [k.text.strip() for k in wiersz.find_all(['td', 'th'])]
        if len(komorki) >= 2:
            ceny[komorki[0].upper()] = na_float(komorki[1])

    # Tabela 2: oplaty za postoj
    postoj = {}
    for wiersz in tabele[1].find_all('tr')[1:]:
        komorki = [k.text.strip() for k in wiersz.find_all(['td', 'th'])]
        if len(komorki) >= 2:
            postoj[komorki[0].upper()] = komorki[1].replace('zl/min', 'PLN/min')

    # Darmowy czas: pierwsze dwie liczby minut z calej strony
    wszystkie_minuty = re.findall(r'(\d+)\s*min', soup.get_text(), re.I)
    minuty = sorted(int(m) for m in wszystkie_minuty)
    dc_free = str(minuty[0]) + ' min' if minuty else '0 min'
    ac_free = str(minuty[1]) + ' min' if len(minuty) > 1 else '0 min'

    return [{
        'Operator': 'ORLEN Charge',
        'Pakiet/Plan': (soup.find('h1') or soup.title).text.strip(),
        'Oplata miesieczna': 0.0,
        'Stawka AC': ceny.get('AC', 0.0),
        'Darmowy czas AC': ac_free,
        'Postoj AC': postoj.get('AC', 'brak danych'),
        'Stawka DC': ceny.get('DC', 0.0),
        'Darmowy czas DC': dc_free,
        'Postoj DC': postoj.get('DC', 'brak danych'),
        'Data pobrania': DATA_POBRANIA,
        'Zrodlo': url,
    }]


# ==========================================
# 2. GREENWAY
# Zrodlo: greenwaynetwork.com/pl/cennik (artykuly HTML)
# Darmowy czas i postoj = hardkodowane (brak na stronie).
# ==========================================
PLANY_GW = ['Energia MAX', 'Energia PLUS', 'Energia STANDARD']

def scrape_greenway():
    url = 'https://greenwaynetwork.com/pl/cennik'
    soup = get_soup(url)
    wyniki = []

    for artykul in soup.find_all('article'):
        tekst = artykul.get_text()

        # Sprawdz czy artykul dotyczy jednego z planow
        plan = None
        for p in PLANY_GW:
            if p.lower() in tekst.lower():
                plan = p
                break
        if plan is None:
            continue

        # Deduplikacja
        if any(w['Pakiet/Plan'] == plan.upper() for w in wyniki):
            continue

        # Parsuj ceny z kolejnych linii
        linie = [l.strip() for l in tekst.splitlines() if l.strip()]
        oplata = 0.0
        ac = 0.0
        dc = 0.0
        for i, linia in enumerate(linie):
            nastepna = linie[i + 1] if i + 1 < len(linie) else ''
            if 'Op\u0142ata miesi\u0119czna' in linia:
                oplata = na_float(nastepna)
            elif linia == 'AC':
                ac = na_float(nastepna)
            elif linia == 'DC':
                dc = na_float(nastepna)

        wyniki.append({
            'Operator': 'GreenWay',
            'Pakiet/Plan': plan.upper(),
            'Oplata miesieczna': oplata,
            'Stawka AC': ac,
            'Darmowy czas AC': '600 min',
            'Postoj AC': '0,40 PLN/min',
            'Stawka DC': dc,
            'Darmowy czas DC': '60 min',
            'Postoj DC': '0,40 PLN/min',
            'Data pobrania': DATA_POBRANIA,
            'Zrodlo': url,
        })

    return wyniki


# ==========================================
# 3. BUDIMEX MOBILITY
# Zrodlo: budimexmobility.pl/nowy-cennik-ladowania-budimex-mobility/
# DC ma rozne stawki wg mocy - pokazujemy min-max.
# Postoj: AC po 180 min, DC po 60 min.
# ==========================================
def scrape_budimex():
    url = 'https://budimexmobility.pl/nowy-cennik-ladowania-budimex-mobility/'
    soup = get_soup(url)

    ac_ceny = []
    dc_ceny = []

    tabela = soup.find('table')
    if tabela:
        for wiersz in tabela.find_all('tr')[1:]:
            komorki = [td.get_text(strip=True) for td in wiersz.find_all(['td', 'th'])]
            if len(komorki) < 2:
                continue
            typ = komorki[0].upper()
            caly_wiersz = ' '.join(komorki[1:])
            liczby = re.findall(r'(\d[,.]\d+)', caly_wiersz)
            if not liczby:
                continue
            cena = na_float(liczby[0])  # pierwsza liczba = cena standardowa
            if 'AC' in typ:
                ac_ceny.append(cena)
            elif 'DC' in typ:
                dc_ceny.append(cena)

    postoj_ac = '0,40 PLN/min po 180 min'
    postoj_dc = '0,40 PLN/min po 60 min'

    return [{
        'Operator': 'Budimex Mobility',
        'Pakiet/Plan': 'Standardowy (bez abonamentu)',
        'Oplata miesieczna': 0.0,
        'Stawka AC': format_widel(ac_ceny) if ac_ceny else 1.96,
        'Darmowy czas AC': darmowy_czas(postoj_ac),
        'Postoj AC': postoj_ac,
        'Stawka DC': format_widel(dc_ceny) if dc_ceny else '2,35-2,66 PLN/kWh',
        'Darmowy czas DC': darmowy_czas(postoj_dc),
        'Postoj DC': postoj_dc,
        'Data pobrania': DATA_POBRANIA,
        'Zrodlo': url,
    }]


# ==========================================
# 4. NOXO
# Zrodlo: PDF cennika na noxo.energy
# Jesli PDF niedostepny - uzywamy wartosci z cennika 01.04.2026.
# ==========================================
NOXO_PDF_URL = 'https://noxo.energy/wp-content/uploads/2026/03/Cennik-NOXO_2026.04.01.pdf'

def scrape_noxo():
    # Probujemy pobrac i sparsowac PDF
    tekst_pdf = ''
    try:
        try:
            from pypdf import PdfReader
        except ImportError:
            import subprocess, sys
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pypdf', '-q'])
            from pypdf import PdfReader

        odpowiedz = requests.get(NOXO_PDF_URL, headers=HEADERS, timeout=15)
        pdf = PdfReader(io.BytesIO(odpowiedz.content))
        for strona in pdf.pages:
            tekst_pdf += strona.extract_text() or ''
    except Exception as e:
        print(f'  [NOXO] Nie udalo sie pobrac PDF: {e} - uzywam danych z cennika 01.04.2026')

    # Szukamy cen AC i DC w tekscie PDF
    ac_ceny = [na_float(m) for m in re.findall(r'AC[^\n]{0,40}?(\d[,.]\d+)\s*z', tekst_pdf, re.I)]
    dc_ceny = [na_float(m) for m in re.findall(r'DC[^\n]{0,40}?(\d[,.]\d+)\s*z', tekst_pdf, re.I)]

    # Szukamy czasu darmowego postoju DC
    wynik = re.search(r'(\d+)\s*min[^A-Z]{0,20}DC', tekst_pdf, re.I)
    postoj_dc = '0,40 PLN/min po ' + wynik.group(1) + ' min' if wynik else '0,40 PLN/min po 30 min'

    return [{
        'Operator': 'NOXO',
        'Pakiet/Plan': 'Standardowy (bez abonamentu)',
        'Oplata miesieczna': 0.0,
        'Stawka AC': format_widel(ac_ceny) if ac_ceny else 2.19,
        'Darmowy czas AC': 'brak danych',
        'Postoj AC': 'brak danych',
        'Stawka DC': format_widel(dc_ceny) if dc_ceny else 3.39,
        'Darmowy czas DC': darmowy_czas(postoj_dc),
        'Postoj DC': postoj_dc,
        'Data pobrania': DATA_POBRANIA,
        'Zrodlo': NOXO_PDF_URL,
    }]


# ==========================================
# 5. TAURON (eTAURON)
# Zrodlo: etauron.tauron.pl - oficjalny PDF z cennikiem (uchwala Zarzadu)
# UWAGA: Tauron ma DODATKOWO cennik dynamiczny (godzinowy/sezonowy,
# powiazany z cenami gieldowymi energii) - ten skrypt rejestruje TYLKO
# cene bazowa (stala) z oficjalnego cennika PDF, nie probuje odwzorowac
# pelnej zlozonosci taryf dynamicznych. Odnotowane wprost w kolumnie
# Pakiet/Plan.
# ==========================================
TAURON_PDF_URL = 'https://etauron.tauron.pl/-/media/emobility/etauron/cennik-etauron.ashx'

def scrape_tauron():
    try:
        from pypdf import PdfReader
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pypdf', '-q'])
        from pypdf import PdfReader

    odpowiedz = requests.get(TAURON_PDF_URL, headers=HEADERS, timeout=15)
    pdf = PdfReader(io.BytesIO(odpowiedz.content))
    tekst = ''
    for strona in pdf.pages:
        tekst += strona.extract_text() or ''

    # Szukamy dwoch wierszy tabeli: "Niezarejestrowany" i "Zarejestrowany",
    # kazdy z para cen AC/DC w formacie "1,89 zł/kWh"
    wyniki = []
    # UWAGA: "Zarejestrowany" jest podciagiem "Niezarejestrowany" - bez
    # negative lookbehind (?<!Nie) regex lapalby zly wiersz tabeli.
    for typ_uzytkownika, wzor_prefiksu in [
        ('Niezarejestrowany', 'Niezarejestrowany'),
        ('Zarejestrowany', r'(?<!Nie)Zarejestrowany'),
    ]:
        wzorzec = re.search(
            wzor_prefiksu + r'\D*?(\d[,.]\d+)\s*z[lł]/kWh\D*?(\d[,.]\d+)\s*z[lł]/kWh',
            tekst, re.I
        )
        if not wzorzec:
            continue
        ac = na_float(wzorzec.group(1))
        dc = na_float(wzorzec.group(2))
        wyniki.append({
            'Operator': 'Tauron (eTAURON)',
            'Pakiet/Plan': f'{typ_uzytkownika} (cena bazowa - operator ma dodatkowo taryfy dynamiczne godzinowe/sezonowe, nieodwzorowane tutaj)',
            'Oplata miesieczna': 0.0,
            'Stawka AC': ac,
            'Darmowy czas AC': '60 min',
            'Postoj AC': '15 PLN/h po 60 min',
            'Stawka DC': dc,
            'Darmowy czas DC': '60 min',
            'Postoj DC': '15 PLN/h po 60 min',
            'Data pobrania': DATA_POBRANIA,
            'Zrodlo': TAURON_PDF_URL,
        })

    if not wyniki:
        raise ValueError('Nie znaleziono wierszy cennika w PDF Tauron - sprawdz, czy struktura dokumentu sie nie zmienila')
    return wyniki


# ==========================================
# 6. ELEPORT
# Zrodlo: eleport.com/pl/oferta/ (strona HTML z cennikiem)
# DC ma kilka progow mocy - pokazujemy min-max (jak Budimex).
# ==========================================
def scrape_eleport():
    url = 'https://eleport.com/pl/oferta/'
    soup = get_soup(url)
    tekst = soup.get_text()

    # Cena AC - pojedyncza wartosc "AC (do 22 kW) - od 1,89 PLN/kWh"
    ac_wzorzec = re.search(r'AC\s*\([^)]*\)\s*[-–]\s*(?:od\s*)?(\d[,.]\d+)\s*PLN/kWh', tekst, re.I)
    ac = na_float(ac_wzorzec.group(1)) if ac_wzorzec else 0.0

    # Ceny DC - kilka przedzialow mocy, zbieramy wszystkie liczby przy "DC"
    dc_ceny = [na_float(m) for m in re.findall(r'DC\s*\([^)]*\)\s*[-–]\s*(?:od\s*)?(\d[,.]\d+)\s*PLN/kWh', tekst, re.I)]

    # Oplaty za postoj - osobno dla AC i DC, format "po X h ... Y PLN/h"
    postoj_ac_wzorzec = re.search(r'Stacje AC.*?po\s*(\d+)\s*h[^0-9]*(\d+)\s*PLN/h', tekst, re.I | re.S)
    postoj_dc_wzorzec = re.search(r'Stacje DC.*?po\s*(\d+)\s*h[^0-9]*(\d+)\s*PLN/h', tekst, re.I | re.S)

    if postoj_ac_wzorzec:
        darmowy_ac = f'{int(postoj_ac_wzorzec.group(1)) * 60} min'
        postoj_ac = f'{postoj_ac_wzorzec.group(2)} PLN/h po {darmowy_ac}'
    else:
        darmowy_ac, postoj_ac = 'brak danych', 'brak danych'

    if postoj_dc_wzorzec:
        darmowy_dc = f'{int(postoj_dc_wzorzec.group(1)) * 60} min'
        postoj_dc = f'{postoj_dc_wzorzec.group(2)} PLN/h po {darmowy_dc}'
    else:
        darmowy_dc, postoj_dc = 'brak danych', 'brak danych'

    if ac == 0.0 and not dc_ceny:
        raise ValueError('Nie znaleziono cen na stronie Eleport - sprawdz, czy struktura strony sie nie zmienila')

    return [{
        'Operator': 'Eleport',
        'Pakiet/Plan': 'Standardowy (bez abonamentu)',
        'Oplata miesieczna': 0.0,
        'Stawka AC': ac,
        'Darmowy czas AC': darmowy_ac,
        'Postoj AC': postoj_ac,
        'Stawka DC': format_widel(dc_ceny) if dc_ceny else 0.0,
        'Darmowy czas DC': darmowy_dc,
        'Postoj DC': postoj_dc,
        'Data pobrania': DATA_POBRANIA,
        'Zrodlo': url,
    }]


# ==========================================
# 7. MOYA ENERGIA
# Zrodlo: moya-energia.pl/dla-ciebie/ (stabilny URL, nie datowany news)
# UWAGA: bardzo rozbudowana struktura - 5 progow mocy DC (60/120/180/
# 400 kW) + taryfa dzienna/nocna dla AC. Pokazujemy widelki (min-max)
# obejmujace WSZYSTKIE progi/tarify razem, podobnie jak przy Budimex.
# Osobny PDF ze szczegolowym cennikiem per-stacja istnieje, ale ma URL
# zawierajacy date aktualizacji (zmienia sie przy kazdej zmianie cen),
# wiec NIE nadaje sie do trwalego zaszycia w skrypcie - nie uzywamy go.
# ==========================================
def scrape_moya():
    url = 'https://moya-energia.pl/dla-ciebie/'
    soup = get_soup(url)
    tekst = soup.get_text()

    def wyciagnij_wszystkie_ceny(blok):
        wyniki = []
        for fragment in re.findall(r'[\d,.\s/–\-]+z[lł]/kWh', blok):
            wyniki.extend(na_float(m) for m in re.findall(r'\d[,.]\d+', fragment))
        return wyniki

    dopasowanie_ac = re.search(r'AC\s*22\s*kW(.*?)DC', tekst, re.S | re.I)
    ac_ceny = wyciagnij_wszystkie_ceny(dopasowanie_ac.group(1)) if dopasowanie_ac else []

    poz_dc = tekst.find('DC')
    blok_dc = tekst[poz_dc:poz_dc + 800] if poz_dc >= 0 else ''
    dc_ceny = wyciagnij_wszystkie_ceny(blok_dc)

    if not ac_ceny and not dc_ceny:
        raise ValueError('Nie znaleziono cen na stronie MOYA - sprawdz, czy struktura strony sie nie zmienila')

    return [{
        'Operator': 'MOYA energia',
        'Pakiet/Plan': 'Aplikacja Super MOYA (widelki obejmuja wszystkie progi mocy 60-400kW oraz tarify dzienna/nocna - zobacz zrodlo dla pelnej tabeli)',
        'Oplata miesieczna': 0.0,
        'Stawka AC': format_widel(ac_ceny) if ac_ceny else 0.0,
        'Darmowy czas AC': 'brak danych',
        'Postoj AC': 'brak danych',
        'Stawka DC': format_widel(dc_ceny) if dc_ceny else 0.0,
        'Darmowy czas DC': 'brak danych',
        'Postoj DC': 'brak danych',
        'Data pobrania': DATA_POBRANIA,
        'Zrodlo': url,
    }]


# ==========================================
# 5. ZAPIS DO XLSX
# ==========================================

# Para (klucz w slowniku, naglowek w pliku)
KOLUMNY = [
    ('Operator',          'Operator'),
    ('Pakiet/Plan',       'Pakiet/Plan'),
    ('Oplata miesieczna', 'Op\u0142ata miesi\u0119czna'),
    ('Stawka AC',         'Stawka AC'),
    ('Darmowy czas AC',   'Darmowy czas AC'),
    ('Postoj AC',         'Post\u00f3j AC'),
    ('Stawka DC',         'Stawka DC'),
    ('Darmowy czas DC',   'Darmowy czas DC'),
    ('Postoj DC',         'Post\u00f3j DC'),
    ('Data pobrania',     'Data pobrania'),
    ('Zrodlo',            '\u0179r\u00f3d\u0142o'),
]

CZCIONKA    = 'Segoe UI'
KOLOR_CIEMNY = '1B365D'
RAMKA = Border(
    left=Side(style='thin', color='E0E0E0'),
    right=Side(style='thin', color='E0E0E0'),
    top=Side(style='thin', color='E0E0E0'),
    bottom=Side(style='thin', color='E0E0E0')
)

def save_xlsx(dane, nazwa_pliku='Zestawienie_Cennikow_EV.xlsx'):
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = 'Cennik EV'

    # Wiersz 1: tytul
    ostatnia_kol = get_column_letter(len(KOLUMNY))
    ws.merge_cells(f'A1:{ostatnia_kol}1')
    ws['A1'] = 'Zestawienie Cennik\u00f3w Operator\u00f3w \u0141adowania EV'
    ws['A1'].font = Font(name=CZCIONKA, size=16, bold=True, color=KOLOR_CIEMNY)
    ws['A1'].alignment = Alignment(horizontal='left', vertical='center')
    ws.row_dimensions[1].height = 40

    # Wiersz 3: naglowki kolumn
    for nr_kol, (klucz, naglowek) in enumerate(KOLUMNY, 1):
        kom = ws.cell(row=3, column=nr_kol, value=naglowek)
        kom.font = Font(name=CZCIONKA, size=11, bold=True, color='FFFFFF')
        kom.fill = PatternFill(start_color=KOLOR_CIEMNY, end_color=KOLOR_CIEMNY, fill_type='solid')
        kom.alignment = Alignment(horizontal='center', vertical='center')
        kom.border = RAMKA
    ws.row_dimensions[3].height = 28

    # Formatowanie komorek wg kolumny
    FORMAT_PLN     = '#,##0.00" PLN"'
    FORMAT_KWPH    = '#,##0.00" PLN/kWh"'
    WYROWNAJ_PRAWO  = {'Oplata miesieczna', 'Stawka AC', 'Stawka DC'}
    WYROWNAJ_SRODEK = {'Darmowy czas AC', 'Postoj AC', 'Darmowy czas DC', 'Postoj DC', 'Data pobrania'}
    TLO_ZEBRA = PatternFill(start_color='F7F9FB', end_color='F7F9FB', fill_type='solid')

    # Wiersze z danymi (od wiersza 4)
    for nr_wiersza, rekord in enumerate(dane, 4):
        for nr_kol, (klucz, naglowek) in enumerate(KOLUMNY, 1):
            wartosc = rekord[klucz]
            kom = ws.cell(row=nr_wiersza, column=nr_kol, value=wartosc)
            kom.font = Font(name=CZCIONKA, size=11)
            kom.border = RAMKA

            # Format liczbowy tylko gdy wartosc jest liczba (float), nie tekst z widel
            if klucz == 'Oplata miesieczna':
                kom.number_format = FORMAT_PLN
            elif klucz in ('Stawka AC', 'Stawka DC') and isinstance(wartosc, float):
                kom.number_format = FORMAT_KWPH

            if klucz in WYROWNAJ_PRAWO:
                kom.alignment = Alignment(horizontal='right', vertical='center')
            elif klucz in WYROWNAJ_SRODEK:
                kom.alignment = Alignment(horizontal='center', vertical='center')
            else:
                kom.alignment = Alignment(horizontal='left', vertical='center')

            if nr_wiersza % 2 == 0:
                kom.fill = TLO_ZEBRA

        ws.row_dimensions[nr_wiersza].height = 22

    # Autodopasowanie szerokosci kolumn
    for kolumna in ws.iter_cols(min_row=2):
        max_dlugosc = max(len(str(kom.value or '')) for kom in kolumna)
        litera = get_column_letter(kolumna[0].column)
        ws.column_dimensions[litera].width = max(max_dlugosc + 5, 14)

    wb.save(nazwa_pliku)
    print(f'[SUKCES] Zapisano: {nazwa_pliku}')


# ==========================================
# MAIN
# ==========================================
SCRAPERS = [
    (scrape_orlen,    'ORLEN Charge'),
    (scrape_greenway, 'GreenWay'),
    (scrape_budimex,  'Budimex Mobility'),
    (scrape_noxo,     'NOXO'),
    (scrape_tauron,   'Tauron (eTAURON)'),
    (scrape_eleport,  'Eleport'),
    (scrape_moya,     'MOYA energia'),
]

dane = []
for scraper, nazwa in SCRAPERS:
    try:
        wyniki = scraper()
        dane.extend(wyniki)
        print(f'[OK] {nazwa}: {len(wyniki)} rekord(ow)')
    except Exception as e:
        print(f'[BLAD] {nazwa}: {e}')

if dane:
    save_xlsx(dane)
else:
    print('[ALERT] Brak danych.')


# ==========================================
# 5. TAURON (eTauron)
# Zrodlo: oficjalny PDF cennika, etauron.tauron.pl
# UWAGA: Tauron ma DODATKOWO cennik dynamiczny (godzinowy/sezonowy,
# zalezny od cen gieldowych energii) - ponizej rejestrujemy TYLKO
# cene bazowa (stala, dla uzytkownikow zarejestrowanych w aplikacji),
# nie probujemy odwzorowac pelnej zlozonosci taryf dynamicznych.
# ==========================================
TAURON_PDF_URL = 'https://etauron.tauron.pl/-/media/emobility/etauron/cennik-etauron.ashx'

def scrape_tauron():
    tekst_pdf = ''
    try:
        try:
            from pypdf import PdfReader
        except ImportError:
            import subprocess, sys
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pypdf', '-q'])
            from pypdf import PdfReader

        odpowiedz = requests.get(TAURON_PDF_URL, headers=HEADERS, timeout=15)
        pdf = PdfReader(io.BytesIO(odpowiedz.content))
        for strona in pdf.pages:
            tekst_pdf += strona.extract_text() or ''
    except Exception as e:
        print(f'  [TAURON] Nie udalo sie pobrac PDF: {e} - uzywam danych z cennika 04.11.2024')

    # Szukamy wiersza "Zarejestrowany X zl/kWh Y zl/kWh" (cena preferencyjna,
    # dla zarejestrowanych w aplikacji eTAURON)
    wynik = re.search(r'Zarejestrowany\s+([\d,\.]+)\s*z[lł]/kWh\s+([\d,\.]+)\s*z[lł]/kWh', tekst_pdf)
    ac = na_float(wynik.group(1)) if wynik else 1.69
    dc = na_float(wynik.group(2)) if wynik else 2.19

    # Darmowy czas postoju (np. "1h postoju za darmo")
    wynik_wolny = re.search(r'(\d+)\s*h\s*postoju\s*za\s*darmo', tekst_pdf, re.I)
    darmowy_min = str(int(wynik_wolny.group(1)) * 60) + ' min' if wynik_wolny else '60 min'

    # Oplata za przekroczenie (np. "oplata - 15 zl/h")
    wynik_postoj = re.search(r'op[lł]ata\s*[\u2013-]\s*([\d,\.]+)\s*z[lł]/h', tekst_pdf, re.I)
    postoj = (wynik_postoj.group(1).replace('.', ',') + ' PLN/h') if wynik_postoj else '15 PLN/h'

    return [{
        'Operator': 'Tauron (eTauron)',
        'Pakiet/Plan': 'Zarejestrowany w aplikacji (cena bazowa)*',
        'Oplata miesieczna': 0.0,
        'Stawka AC': ac,
        'Darmowy czas AC': darmowy_min,
        'Postoj AC': postoj,
        'Stawka DC': dc,
        'Darmowy czas DC': darmowy_min,
        'Postoj DC': postoj,
        'Data pobrania': DATA_POBRANIA,
        'Zrodlo': TAURON_PDF_URL,
    }]


# ==========================================
# 6. ELEPORT
# Zrodlo: eleport.com/pl/oferta/ (strona HTML)
# DC ma 3 progi mocy (50-99kW, 100-149kW, 150kW+) - pokazujemy widelki.
# ==========================================
def scrape_eleport():
    url = 'https://eleport.com/pl/oferta/'
    soup = get_soup(url)
    caly_tekst = soup.get_text(' ', strip=True)

    ac_ceny = [na_float(m) for m in re.findall(r'AC[^\d]{0,25}(\d[,.]\d+)\s*PLN/kWh', caly_tekst)]
    dc_ceny = [na_float(m) for m in re.findall(r'DC[^\d]{0,25}(\d[,.]\d+)\s*PLN/kWh', caly_tekst)]

    # Postoj AC: "Po 4 h ... 24 PLN/h" | Postoj DC: "Po 1 h ... 33 PLN/h"
    postoj_ac_match = re.search(r'Stacje AC.{0,60}?Po\s*(\d+)\s*h.{0,40}?(\d+)\s*PLN/h', caly_tekst, re.I | re.S)
    postoj_dc_match = re.search(r'Stacje DC.{0,60}?Po\s*(\d+)\s*h.{0,40}?(\d+)\s*PLN/h', caly_tekst, re.I | re.S)

    darmowy_ac = str(int(postoj_ac_match.group(1)) * 60) + ' min' if postoj_ac_match else '240 min'
    postoj_ac = postoj_ac_match.group(2) + ' PLN/h' if postoj_ac_match else '24 PLN/h'
    darmowy_dc = str(int(postoj_dc_match.group(1)) * 60) + ' min' if postoj_dc_match else '60 min'
    postoj_dc = postoj_dc_match.group(2) + ' PLN/h' if postoj_dc_match else '33 PLN/h'

    return [{
        'Operator': 'Eleport',
        'Pakiet/Plan': 'Standardowy (bez abonamentu)',
        'Oplata miesieczna': 0.0,
        'Stawka AC': format_widel(ac_ceny) if ac_ceny else 1.89,
        'Darmowy czas AC': darmowy_ac,
        'Postoj AC': postoj_ac,
        'Stawka DC': format_widel(dc_ceny) if dc_ceny else '2,95-3,33 PLN/kWh',
        'Darmowy czas DC': darmowy_dc,
        'Postoj DC': postoj_dc,
        'Data pobrania': DATA_POBRANIA,
        'Zrodlo': url,
    }]


[OK] ORLEN Charge: 1 rekord(ow)
[OK] GreenWay: 3 rekord(ow)
[OK] Budimex Mobility: 1 rekord(ow)
[OK] NOXO: 1 rekord(ow)
[OK] Tauron (eTAURON): 2 rekord(ow)
[BLAD] Eleport: Nie znaleziono cen na stronie Eleport - sprawdz, czy struktura strony sie nie zmienila
[OK] MOYA energia: 1 rekord(ow)
[SUKCES] Zapisano: Zestawienie_Cennikow_EV.xlsx
